# 🧠 Week 9: Kubernetes Orchestration, Queue-Based Autoscaling & Unit Economics
## KEDA Custom Metrics, Why GPU% HPA Fails & The Cost per Million Tokens Mathematical Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simranjeet97/AI_Engineer_Roadmap_2027/blob/main/notebooks/week_09_kubernetes_keda_and_unit_economics.ipynb)
[![Curriculum](https://img.shields.io/badge/Curriculum-10%20Weeks%20%7C%2050%20Sessions-blue.svg)](https://github.com/simranjeet97/AI_Engineer_Roadmap_2027)
[![Author](https://img.shields.io/badge/Author-Simranjeet%20Singh-black.svg)](https://github.com/simranjeet97)

---

### Staff Engineer Operational Thesis
> *"An inference architecture that is technically brilliant but economically upside-down is an engineering failure.*
>
> *In production AI systems, infrastructure costs can bankrupt a company before product-market fit is achieved. Traditional cloud infrastructure practices fail catastrophically when applied to LLMs. Autoscaling on CPU or GPU utilization is an amateur mistake: modern inference engines pre-allocate 90% of GPU VRAM at boot for PagedAttention pools and spin in busy polling loops, appearing '100% utilized' even when zero requests are running.*
>
> *Staff AI Engineers autoscale strictly on **Queue Depth** and **KV Cache Saturation** via **KEDA (Kubernetes Event-driven Autoscaling)**. And they evaluate every system design decision against a relentless unit economics ledger: the cost per million tokens ($/Mtok) and cluster breakeven volume."*

### 🎯 Week 9 Learning Objectives:
1. **Kubernetes for AI Workloads**: Master the NVIDIA GPU Operator, Container Device Interface (CDI), and IPC shared memory requirements (`/dev/shm`).
2. **The GPU% Autoscaling Trap**: Prove why default Horizontal Pod Autoscaler (HPA) fails on LLM engines and burns hundreds of thousands of dollars on idle GPUs.
3. **Queue-Depth Autoscaling with KEDA**: Write production `ScaledObject` manifests triggering pod scale-out based on Prometheus queue metrics.
4. **The Unit Economics of Inference**: Formulate the Total Cost of Ownership (TCO) down to the exact dollar cost per million output tokens ($/Mtok).
5. **The Capacity Planning Workbook**: Model diurnal traffic curves and size GPU clusters to guarantee 99.9% SLO compliance while minimizing idle waste.


---
## 🛠️ Step 0: Environment Setup & Infrastructure Dependencies

We initialize our simulation environment with tools for economic modeling, Kubernetes manifest validation, and diurnal timeseries generation.


In [ ]:
# @title Step 0: Environment Setup & Infrastructure Dependencies
import os
import sys
import math
import time
import random
from typing import Dict, List, Any, Optional, Tuple

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    !pip install -q numpy pandas matplotlib seaborn
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

print("✅ Week 9 infrastructure & economics environment initialized.")


---
## ☸️ Day 41: Kubernetes for AI Workloads: GPU Operator & DaemonSets

### The Kubernetes AI Infrastructure Stack
To run LLM inference on Kubernetes, traditional container runtimes are insufficient:
1. **NVIDIA GPU Operator**: Automates driver installation, container runtime configuration, and CUDA library injection onto worker nodes.
2. **Container Device Interface (CDI)**: Standardized specification allowing container engines (containerd/CRI-O) to bind physical GPU devices.
3. **The Shared Memory Trap (`/dev/shm`)**:
   * PyTorch distributed engines (Ray, vLLM, TensorRT-LLM) use POSIX shared memory for ultra-fast inter-process communication (IPC) between tensor-parallel workers.
   * Default Kubernetes containers provide a tiny **64 MB** `/dev/shm` mount!
   * Running vLLM on a default pod triggers an immediate unrecoverable crash: `Bus error (core dumped)`.
   * **Mandatory Fix**: Mount a memory-backed volume `emptyDir: { medium: Memory }` to `/dev/shm`.

Let's generate and validate a production Kubernetes Deployment manifest for an inference engine.


In [ ]:
# @title Day 41: Production Kubernetes Manifest Generator & Validator
def generate_production_k8s_manifest(
    model_name: str = "meta-llama/Meta-Llama-3-8B-Instruct",
    num_gpus: int = 1,
    port: int = 8000
) -> str:
    """
    Generates a production-grade Kubernetes Deployment YAML for vLLM
    with proper GPU resource limits, IPC shared memory mounts, and health probes.
    """
    manifest = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: apex-infer-{model_name.split('/')[-1].lower()}
  namespace: inference-prod
  labels:
    app: apex-infer
    model: {model_name.split('/')[-1].lower()}
spec:
  replicas: 2
  selector:
    matchLabels:
      app: apex-infer
  template:
    metadata:
      labels:
        app: apex-infer
      annotations:
        prometheus.io/scrape: "true"
        prometheus.io/port: "{port}"
        prometheus.io/path: "/metrics"
    spec:
      tolerations:
      - key: "nvidia.com/gpu"
        operator: "Exists"
        effect: "NoSchedule"
      containers:
      - name: inference-engine
        image: vllm/vllm-openai:latest
        args:
        - "--model"
        - "{model_name}"
        - "--tensor-parallel-size"
        - "{num_gpus}"
        - "--enable-prefix-caching"
        - "--enable-chunked-prefill"
        - "--max-num-batched-tokens"
        - "512"
        - "--port"
        - "{port}"
        resources:
          limits:
            nvidia.com/gpu: "{num_gpus}"
            memory: "32Gi"
            cpu: "8"
          requests:
            nvidia.com/gpu: "{num_gpus}"
            memory: "16Gi"
            cpu: "4"
        volumeMounts:
        - mountPath: /dev/shm
          name: dshm
        readinessProbe:
          httpGet:
            path: /health
            port: {port}
          initialDelaySeconds: 45
          periodSeconds: 5
        livenessProbe:
          httpGet:
            path: /health
            port: {port}
          initialDelaySeconds: 60
          periodSeconds: 15
      volumes:
      - name: dshm
        emptyDir:
          medium: Memory
          sizeLimit: "16Gi"
"""
    return manifest

yaml_spec = generate_production_k8s_manifest()
print("=== PRODUCTION KUBERNETES MANIFEST (apex-infer.yaml) ===")
print(yaml_spec)


### 🔍 Deep Dive: Key Architectural Directives
1. **`emptyDir: { medium: Memory, sizeLimit: "16Gi" }`**: Mounts 16GB of host RAM as `/dev/shm`, preventing PyTorch IPC tensor-parallel communication crashes.
2. **`tolerations: [{ key: "nvidia.com/gpu" }]`**: Ensures pods schedule exclusively on dedicated GPU-accelerated node pools.
3. **`prometheus.io/scrape: "true"`**: Exposes internal telemetry directly to Prometheus for KEDA autoscaling.


---
## 💸 Day 42: Why CPU/GPU Utilization Autoscaling Destroys Unit Economics

### The Two Fallacies of GPU Metric Autoscaling
Standard Kubernetes autoscaling uses Horizontal Pod Autoscaler (HPA) targeting CPU or GPU utilization (e.g., `avg_utilization > 70%`). For LLM inference engines, this is **completely broken**:

```
1. THE VRAM PRE-ALLOCATION FALLACY:
   - At startup, vLLM / SGLang pre-allocates ~90% of GPU memory for its PagedAttention KV pool.
   - Traditional metrics tools (like nvidia-smi / DCGM) report: "GPU Memory: 90% USED!"
   - Standard HPA thinks the pod is overloaded and immediately triggers a scale-out!

2. THE BUSY-POLLING SM FALLACY:
   - Modern continuous batching runtimes use tight polling loops to avoid CUDA kernel launch latency.
   - Even when 0 user requests are active, the GPU Streaming Multiprocessors report 80-100% compute load.
   - Standard HPA keeps the cluster permanently scaled to maximum replicas 24/7!
```

Let's simulate a 24-hour diurnal traffic curve comparing **Standard GPU% Autoscaling** against **KEDA Queue-Depth Autoscaling**.


In [ ]:
# @title Day 42: Autoscaling Simulation: Standard HPA vs KEDA Queue-Depth
def simulate_24h_autoscaling_comparison():
    hours = np.linspace(0, 24, 240) # 24 hours (sample every 6 mins)
    
    # Diurnal traffic curve: Low at night (2 req/s), peak at midday (60 req/s)
    traffic_req_s = 25.0 * np.sin((hours - 6) / 24 * 2 * np.pi) + 32.0
    traffic_req_s = np.clip(traffic_req_s + np.random.normal(0, 2, len(hours)), 2, 80)
    
    # 1. Standard HPA (scaling on GPU% utilization)
    # Because GPU reports ~85% utilization constantly, HPA is pegged at max pods (8 pods) 24/7!
    hpa_pods = np.full(len(hours), 8)
    
    # 2. KEDA Queue-Depth Autoscaling
    # Each pod comfortably handles 10 requests/s before queue depth exceeds threshold
    keda_pods = np.ceil(traffic_req_s / 10.0).astype(int)
    keda_pods = np.clip(keda_pods, 1, 8) # Min 1 pod, Max 8 pods
    
    # Financial Calculation: H100 SXM5 cost = $3.50 / hour per pod
    hourly_rate = 3.50
    hpa_cost_24h = np.sum(hpa_pods * (24.0 / len(hours)) * hourly_rate)
    keda_cost_24h = np.sum(keda_pods * (24.0 / len(hours)) * hourly_rate)
    savings_pct = (hpa_cost_24h - keda_cost_24h) / hpa_cost_24h * 100.0
    
    # Plot Comparison
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), dpi=150, sharex=True)
    
    # Traffic Curve
    ax1.plot(hours, traffic_req_s, color='black', linewidth=2, label="Incoming Diurnal Traffic (req/s)")
    ax1.set_title("24-Hour Production Traffic Curve & Autoscaler Pod Allocation", fontsize=13, fontweight='bold')
    ax1.set_ylabel("Traffic (Requests / Sec)", fontweight='bold')
    ax1.legend(loc="upper right", frameon=True)
    
    # Pod Allocations
    ax2.plot(hours, hpa_pods, color='crimson', linestyle='--', linewidth=2.5, label=f"Standard HPA (GPU% Metric) - 24h Cost: ${hpa_cost_24h:.2f}")
    ax2.step(hours, keda_pods, color='teal', linewidth=2.5, label=f"KEDA Queue-Depth Autoscaler - 24h Cost: ${keda_cost_24h:.2f}")
    ax2.set_title(f"Cluster Pod Scaling Efficiency: KEDA Achieves {savings_pct:.1f}% Cost Reduction", fontsize=12, fontweight='bold')
    ax2.set_xlabel("Time of Day (Hours)", fontweight='bold')
    ax2.set_ylabel("Active GPU Pods", fontweight='bold')
    ax2.set_ylim(0, 10)
    ax2.legend(loc="upper right", frameon=True)
    
    plt.tight_layout()
    plt.show()

simulate_24h_autoscaling_comparison()


### 🔍 Deep Dive: What Did the Day 42 Code Prove?
* **Standard HPA (Crimson)**: Kept 8 $\times$ H100 GPUs running continuously overnight, racking up **$672.00 / day**.
* **KEDA Queue-Depth (Teal)**: Scaled down to 1 pod during off-peak hours and scaled up to 8 pods only during peak traffic, spending **$301.00 / day**.
* **The Savings**: An instant **$55.2\%$ infrastructure cost reduction** simply by fixing the autoscaling trigger metric!


---
## 🎯 Day 43: Autoscaling on Queue Depth with KEDA & Prometheus

### The KEDA ScaledObject Architecture
KEDA queries your Prometheus server directly every $10\text{s}$ and evaluates user-defined PromQL queries against target thresholds:
* **Primary Metric**: `vllm:num_requests_waiting` (Scale up when queue depth $> 8$ for $> 30\text{s}$).
* **Secondary Metric**: `vllm:gpu_cache_usage_factor` (Scale up when KV memory exceeds $85\%$, indicating imminent preemption).
* **Stabilization Window**: Enforces a **300-second cooldown** on scale-down events to prevent pod thrashing / flapping.

Let's generate the production KEDA manifest.


In [ ]:
# @title Day 43: Production KEDA ScaledObject Manifest Generator
def generate_keda_scaled_object(
    deployment_name: str = "apex-infer-meta-llama-3-8b-instruct",
    prometheus_url: str = "http://prometheus-k8s.monitoring.svc:9090",
    min_replicas: int = 1,
    max_replicas: int = 10
) -> str:
    manifest = f"""apiVersion: keda.sh/v1alpha1
kind: ScaledObject
metadata:
  name: {deployment_name}-autoscaler
  namespace: inference-prod
spec:
  scaleTargetRef:
    name: {deployment_name}
  minReplicaCount: {min_replicas}
  maxReplicaCount: {max_replicas}
  cooldownPeriod: 300
  pollingInterval: 10
  advanced:
    horizontalPodAutoscalerConfig:
      behavior:
        scaleDown:
          stabilizationWindowSeconds: 300
          policies:
          - type: Percent
            value: 20
            periodSeconds: 60
  triggers:
  # Trigger 1: Queue Depth (Waiting Requests)
  - type: prometheus
    metadata:
      serverAddress: {prometheus_url}
      metricName: vllm_num_requests_waiting
      query: sum(vllm:num_requests_waiting{{app="{deployment_name.split('-')[0]}"}})
      threshold: "8"
  # Trigger 2: GPU KV Cache Saturation
  - type: prometheus
    metadata:
      serverAddress: {prometheus_url}
      metricName: vllm_gpu_cache_usage_factor
      query: avg(vllm:gpu_cache_usage_factor{{app="{deployment_name.split('-')[0]}"}})
      threshold: "0.85"
"""
    return manifest

keda_manifest = generate_keda_scaled_object()
print("=== PRODUCTION KEDA SCALEDOBJECT (keda-scaledobject.yaml) ===")
print(keda_manifest)


### 🔍 Deep Dive: Key KEDA Directives
1. **`threshold: "8"` on `vllm:num_requests_waiting`**: Tells KEDA: "For every 8 requests waiting in the scheduler queue, add 1 additional GPU pod."
2. **`stabilizationWindowSeconds: 300`**: Prevents the autoscaler from terminating GPU pods immediately when a transient traffic dip occurs, preserving warm prefix caches!


---
## 💰 Day 44: The Unit Economics of Inference: Cost per Million Tokens

### The Master Inference Cost Formula
How do you calculate whether your self-hosted LLM service is financially viable?

$$\text{Cost per 1M Output Tokens (\$/Mtok)} = \frac{\text{Hourly GPU Server Cost (\$/hr)}}{\text{Effective Output Tokens Generated per Hour}} \times 1,000,000$$

Where effective output tokens generated per hour is:
$$\text{Tokens / Hour} = \text{Sustained Throughput (tok/s)} \times 3,600 \times \text{Cluster Utilization Factor } (\eta)$$

* If an NVIDIA H100 SXM5 node costs **$3.50 / hour** and generates a sustained **$120 \text{ tok/s}$** at $70\%$ utilization:
  $$\text{Effective Tokens / Hour} = 120 \times 3600 \times 0.70 = 302,400 \text{ tokens/hr}$$
  $$\text{Cost per 1M Tokens} = \frac{\$3.50}{302,400} \times 1,000,000 = \mathbf{\$11.57 / \text{Mtok}}$$

Let's build a financial breakeven calculator comparing self-hosting vs commercial API providers (e.g., GPT-4o, Claude 3.5 Sonnet).


In [ ]:
# @title Day 44: Self-Hosting vs Managed API Breakeven Financial Calculator
def calculate_unit_economics_breakeven():
    # Cloud GPU options (Hourly on-demand pricing)
    gpu_options = {
        "NVIDIA H100 SXM5 (80GB)": {"hourly_cost": 3.50, "sustained_tok_s": 140.0},
        "NVIDIA A100 SXM4 (80GB)": {"hourly_cost": 2.20, "sustained_tok_s": 75.0},
        "NVIDIA L40S (48GB)":      {"hourly_cost": 1.40, "sustained_tok_s": 50.0}
    }
    
    # Commercial Managed API pricing per 1M Output Tokens
    api_benchmarks = {
        "Commercial Tier-1 (GPT-4o)": 10.00,        # $10.00 / 1M output
        "Commercial Tier-2 (Claude 3.5 Sonnet)": 15.00, # $15.00 / 1M output
        "Commercial Fast (Llama-3-70B Hosted API)": 0.80 # $0.80 / 1M output
    }
    
    utilization_factors = [0.30, 0.50, 0.70, 0.90] # 30% to 90% cluster utilization
    
    records = []
    for gpu_name, spec in gpu_options.items():
        cost_hr = spec["hourly_cost"]
        tok_s = spec["sustained_tok_s"]
        
        for u in utilization_factors:
            eff_tokens_hr = tok_s * 3600 * u
            cost_per_1m = (cost_hr / eff_tokens_hr) * 1e6
            
            records.append({
                "GPU Accelerator": gpu_name.split(" ")[1],
                "Cluster Utilization": f"{int(u*100)}%",
                "Tokens / Hour": f"{int(eff_tokens_hr):,}",
                "Cost per 1M Tokens ($)": round(cost_per_1m, 2),
                "Monthly Cost (24/7)": f"${cost_hr * 730:.0f}"
            })
            
    df_econ = pd.DataFrame(records)
    return df_econ

df_unit_economics = calculate_unit_economics_breakeven()
print("=== SELF-HOSTED INFERENCE UNIT ECONOMICS ($ / 1M TOKENS) ===")
display(df_unit_economics.pivot(index="Cluster Utilization", columns="GPU Accelerator", values="Cost per 1M Tokens ($)"))


### 🔍 Deep Dive: The Law of Cluster Utilization
Look at the table above:
* On an **H100 SXM5**:
  * At **$30\%$ utilization** (poor traffic engineering, low batching): Cost is **$23.15 / \text{Mtok}** (more expensive than commercial APIs!).
  * At **$90\%$ utilization** (continuous batching, prefix caching, KEDA autoscaling): Cost plummets to **$7.72 / \text{Mtok}**!

> **Staff Engineer Rule**: Hardware efficiency is financial efficiency. If your cluster operates at $< 40\%$ average utilization, self-hosting is financial malpractice. Continuous batching and KEDA autoscaling are what make self-hosting profitable.


---
## 📈 Day 45: The Capacity Planning Workbook

### Sizing the Cluster for Peak Diurnal Demand
How do you size an inference fleet to handle a diurnal peak of $10,000 \text{ req/min}$ while honoring a strict **P99 TTFT SLA of $< 300\text{ms}$**?

We simulate:
1. Hourly peak traffic distribution.
2. Minimum GPU instances required per hour to guarantee queue depth $< 10$.
3. Total cluster budget optimization.


In [ ]:
# @title Day 45: The Production Capacity Planning Visualizer
def plan_cluster_capacity():
    hours = np.arange(0, 24, 1)
    # Traffic profile: requests per minute throughout the day
    traffic_rpm = 1500 + 7500 * np.exp(-((hours - 14)**2) / 18.0) # Peak at 2 PM (14:00)
    
    # Single GPU capacity: handles 400 requests/minute while maintaining P99 TTFT < 300ms
    gpu_capacity_rpm = 400.0
    
    # Required GPUs per hour
    required_gpus = np.ceil(traffic_rpm / gpu_capacity_rpm).astype(int)
    
    fig, ax1 = plt.subplots(figsize=(12, 5), dpi=150)
    
    ax1.plot(hours, traffic_rpm, color='royalblue', linewidth=2.5, marker='o', label="Incoming Traffic (Req / Min)")
    ax1.set_xlabel("Hour of Day (UTC)", fontweight='bold')
    ax1.set_ylabel("Traffic (Requests / Minute)", color='royalblue', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    ax2 = ax1.twinx()
    ax2.step(hours, required_gpus, color='crimson', linewidth=2.5, where='mid', label="Required H100 GPUs (KEDA Target)")
    ax2.set_ylabel("Active H100 GPUs Required", color='crimson', fontweight='bold')
    ax2.set_ylim(0, max(required_gpus) + 5)
    
    plt.title("Production Capacity Planner: Peak Diurnal Sizing & GPU Provisioning", fontsize=13, fontweight='bold')
    fig.tight_layout()
    plt.show()

plan_cluster_capacity()


### 🎯 Week 9 Staff Takeaways & Synthesis Checklist

1. **Never Autoscale on CPU/GPU %**: Inference engines pre-allocate VRAM and spin in tight CUDA polling loops. Traditional HPA keeps your cluster permanently overprovisioned.
2. **KEDA Queue-Depth Autoscaling**: Autoscale strictly on `vllm:num_requests_waiting` and `vllm:gpu_cache_usage_factor` with a 300-second stabilization cooldown window.
3. **Mount `/dev/shm` as Memory**: Always configure `emptyDir: { medium: Memory }` on Kubernetes pods to prevent PyTorch IPC shared memory bus errors.
4. **Utilization Dictates Unit Economics**: Self-hosting costs scale inversely with cluster utilization. High utilization ($> 70\%$) cuts cost-per-million-tokens by more than half.

---

## 📝 Practice Exercises (Test Your Understanding)

### Exercise 1: Self-Hosting vs Managed API Breakeven Volume Analysis
Your company currently spends **$50,000 / month** querying a commercial API provider at **$10.00 / 1M output tokens**.
* You evaluate hosting Llama-3-70B internally on leased 8 $	imes$ H100 SXM5 nodes.
* Each node costs **$22,000 / month** (all-inclusive lease, power, networking, and support).
* A single 8-GPU node generates a sustained **$600 \text{ tokens/sec}$** at $65\%$ average utilization.
* **Question**: How many tokens per month does the company consume currently? How many monthly tokens can a single 8-GPU node deliver? Is self-hosting economically viable at this volume? What is the net monthly savings or loss?

### Exercise 2: KEDA Autoscaling Reaction Time & Cold Start Latency Penalty
An inference pod running Llama-3-70B has a **cold-start startup time of 90 seconds** (loading 70GB of weights from container storage into GPU HBM).
* Traffic surges from $100 \text{ req/min}$ to $1,000 \text{ req/min}$ in 10 seconds.
* KEDA triggers a scale-out from 1 replica to 3 replicas.
* **Question**: What happens to requests arriving during the 90-second cold-start window? How large does the queue depth grow? How does an infrastructure architect mitigate this cold-start penalty (e.g., warm standby pods, model weight caching with JuiceFS / NVMe)?
